# FotMob Bundesliga Tipico Odds Extractor

This notebook prompts for a Bundesliga matchday, reads `match_ids_{matchday}_fotmob.json` from its configured source output directory, and retrieves full-time 1X2 decimal odds from FotMob for every listed match.

It uses one reusable Selenium session with `undetected_chromedriver` and the FotMob `matchOdds` endpoint configured for Germany and the single bookmaker `Tipico_Germany`. Each input match remains in the result even when odds are unavailable; problems are recorded in that match's `issues` list.

The primary output is `matchday_{matchday}_odds_fotmob.json`. Each match contains its ID and team names, zero or one Tipico bookmaker record, and any retrieval or validation issues.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    FOTMOB_MATCH_IDS_DIR,
    FOTMOB_ODDS_DIR,
    ensure_directory,
)


## Imports and notebook constants

Load the JSON, path, browser, and validation utilities. The endpoint is fixed to the German Tipico provider and decimal 1X2 odds.


In [2]:
# Import the libraries required by this notebook step.
import json
import math
from pathlib import Path
from pprint import pprint
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Browser dependencies are missing. Install them in this Jupyter kernel with "
        "'%pip install undetected-chromedriver selenium', then restart the kernel."
    ) from exc


# Set workflow configuration value: TIPICO_PERSISTENT_KEY.
TIPICO_PERSISTENT_KEY = "Tipico_Germany"
# Set workflow configuration value: TIPICO_PROVIDER_ID.
TIPICO_PROVIDER_ID = 171
# Set workflow configuration value: TIPICO_DISPLAY_NAME.
TIPICO_DISPLAY_NAME = "Tipico"
# Set workflow configuration value: COUNTRY_CODE.
COUNTRY_CODE = "DEU"
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20
# Set workflow configuration value: ENDPOINT_TEMPLATE.
ENDPOINT_TEMPLATE = (
    "https://www.fotmob.com/api/data/matchOdds"
    "?matchId={match_id}&ccode3=DEU&bettingProvider=Tipico_Germany"
)


## Select the matchday and resolve files

Ask for the actual Bundesliga matchday, require a positive integer, and construct current-directory input and output paths.


In [3]:
# Handle for matchday for reuse in the workflow.
def prompt_for_matchday() -> int:
    """Read and validate the Bundesliga matchday."""
    raw_value = input("Enter Bundesliga matchday: ").strip()
    # Handle expected failures with a clear, actionable message.
    try:
        selected_matchday = int(raw_value)
    except ValueError as exc:
        raise ValueError(
            "Bundesliga matchday must be entered as a whole number."
        ) from exc

    # Validate the input before continuing with later processing.
    if selected_matchday < 1:
        raise ValueError("Bundesliga matchday must be at least 1.")
    return selected_matchday


matchday = prompt_for_matchday()
input_path = FOTMOB_MATCH_IDS_DIR / f"match_ids_{matchday}_fotmob.json"
output_path = ensure_directory(FOTMOB_ODDS_DIR) / f"matchday_{matchday}_odds_fotmob.json"

print(f"Project root: {_PROJECT_ROOT}")
print(f"Input file:       {input_path}")
print(f"Output file:      {output_path}")


Enter Bundesliga matchday:  1


Project root: C:\kickbase project
Input file:       C:\kickbase project\outputs\fotmob\match_ids\match_ids_1_fotmob.json
Output file:      C:\kickbase project\outputs\fotmob\odds\matchday_1_odds_fotmob.json


## Load and validate the match list

Read UTF-8 JSON and require a non-empty top-level list. Only the match ID and home/away names are carried forward; team IDs are not part of the odds output.


In [4]:
# Parse and validate input json for reuse in the workflow.
def parse_input_json(raw_text: str, source_path: Path) -> Any:
    """Parse input text as JSON with a location-aware error."""
    # Handle expected failures with a clear, actionable message.
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Input file is not valid JSON (line {exc.lineno}, "
            f"column {exc.colno}): {source_path}"
        ) from exc


# Validate input matches for reuse in the workflow.
def validate_input_matches(raw_matches: Any) -> list[dict[str, Any]]:
    """Validate and normalize the input match list."""
    # Validate the input before continuing with later processing.
    if not isinstance(raw_matches, list):
        raise ValueError("The input JSON must contain a top-level list of matches.")
    # Validate the input before continuing with later processing.
    if not raw_matches:
        raise ValueError("The input JSON match list must not be empty.")

    validated_matches: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for index, item in enumerate(raw_matches, start=1):
        # Validate the input before continuing with later processing.
        if not isinstance(item, dict):
            raise ValueError(f"Input item {index} must be a JSON object.")

        match_id = item.get("match_id")
        home_team = item.get("home_team")
        away_team = item.get("away_team")

        # Validate the input before continuing with later processing.
        if (
            not isinstance(match_id, int)
            or isinstance(match_id, bool)
            or match_id < 1
        ):
            raise ValueError(
                f"Input item {index} has no valid positive integer match_id."
            )
        # Validate the input before continuing with later processing.
        if not isinstance(home_team, str) or not home_team.strip():
            raise ValueError(f"Input item {index} has no valid home_team.")
        # Validate the input before continuing with later processing.
        if not isinstance(away_team, str) or not away_team.strip():
            raise ValueError(f"Input item {index} has no valid away_team.")

        validated_matches.append(
            {
                "match_id": match_id,
                "home_team": home_team.strip(),
                "away_team": away_team.strip(),
            }
        )

    return validated_matches


# Load input matches for reuse in the workflow.
def load_input_matches(source_path: Path) -> list[dict[str, Any]]:
    """Read, parse, and validate a FotMob match-ID file."""
    # Handle expected failures with a clear, actionable message.
    try:
        raw_text = source_path.read_text(encoding="utf-8")
    except FileNotFoundError as exc:
        raise FileNotFoundError(
            f"Input file not found: {source_path}. "
            "Run the FotMob match-ID notebook for the selected matchday."
        ) from exc
    except UnicodeDecodeError as exc:
        raise ValueError(f"Input file is not valid UTF-8: {source_path}") from exc
    except OSError as exc:
        raise OSError(f"Could not read input file {source_path}: {exc}") from exc

    return validate_input_matches(parse_input_json(raw_text, source_path))


matches = load_input_matches(input_path)
print(f"Loaded and validated {len(matches)} match(es).")


Loaded and validated 9 match(es).


## Validate the provider and extract Tipico 1X2 odds

Use `odds.matchfactMarkets` as the canonical market source. If it has no 1X2 market, search the nested `odds.oddsTabMarkets` list as a fallback. Only selections `1`, `x`, and `2` are exported.


In [5]:
# Define Odds Notebook Error to keep related behaviour explicit.
class OddsNotebookError(RuntimeError):
    """Base class for expected odds-processing failures."""


# Define Odds Retrieval Error to keep related behaviour explicit.
class OddsRetrievalError(OddsNotebookError):
    """Raised when an odds response cannot be retrieved."""


# Define Odds Not Found Error to keep related behaviour explicit.
class OddsNotFoundError(OddsNotebookError):
    """Raised when FotMob reports that no endpoint resource exists."""


# Define Invalid Odds JSONError to keep related behaviour explicit.
class InvalidOddsJSONError(OddsNotebookError):
    """Raised when FotMob does not return the expected JSON object."""


# Define Odds Unavailable Error to keep related behaviour explicit.
class OddsUnavailableError(OddsNotebookError):
    """Raised when valid Tipico 1X2 odds are unavailable."""


# Normalize provider ID for reuse in the workflow.
def normalize_provider_id(value: Any, field_name: str) -> int:
    """Convert a provider ID to a positive integer."""
    # Validate the input before continuing with later processing.
    if isinstance(value, bool):
        raise OddsUnavailableError(f"{field_name} must be a numeric provider ID.")
    # Validate the input before continuing with later processing.
    if isinstance(value, int):
        provider_id = value
    # Validate the input before continuing with later processing.
    elif isinstance(value, str) and value.strip().isdigit():
        provider_id = int(value.strip())
    else:
        raise OddsUnavailableError(f"{field_name} must be a numeric provider ID.")

    # Validate the input before continuing with later processing.
    if provider_id < 1:
        raise OddsUnavailableError(f"{field_name} must be greater than zero.")
    return provider_id


# Validate tipico payload for reuse in the workflow.
def validate_tipico_payload(
    payload: dict[str, Any],
) -> tuple[dict[str, Any], int]:
    """Require the requested Tipico provider and return its odds object."""
    persistent_key = payload.get("persistentKey")
    # Validate the input before continuing with later processing.
    if persistent_key != TIPICO_PERSISTENT_KEY:
        raise OddsUnavailableError(
            f"Expected provider {TIPICO_PERSISTENT_KEY!r}, "
            f"but received {persistent_key!r}."
        )

    provider_id = TIPICO_PROVIDER_ID
    # Validate the input before continuing with later processing.
    if payload.get("providerId") is not None:
        provider_id = normalize_provider_id(
            payload["providerId"], "Top-level providerId"
        )
        # Validate the input before continuing with later processing.
        if provider_id != TIPICO_PROVIDER_ID:
            raise OddsUnavailableError(
                f"Expected Tipico provider ID {TIPICO_PROVIDER_ID}, "
                f"but received {provider_id}."
            )

    odds = payload.get("odds")
    # Validate the input before continuing with later processing.
    if not isinstance(odds, dict):
        raise OddsUnavailableError("The response has no valid 'odds' object.")

    nested_provider = odds.get("provider")
    # Validate the input before continuing with later processing.
    if nested_provider is not None:
        # Validate the input before continuing with later processing.
        if not isinstance(nested_provider, dict):
            raise OddsUnavailableError(
                "The odds.provider field must be a JSON object when present."
            )
        # Validate the input before continuing with later processing.
        if nested_provider.get("id") is not None:
            nested_id = normalize_provider_id(
                nested_provider["id"], "odds.provider.id"
            )
            # Validate the input before continuing with later processing.
            if nested_id != TIPICO_PROVIDER_ID:
                raise OddsUnavailableError(
                    f"Expected nested Tipico provider ID "
                    f"{TIPICO_PROVIDER_ID}, but received {nested_id}."
                )

    return odds, provider_id


# Check whether 1x2 market for reuse in the workflow.
def is_1x2_market(market: dict[str, Any]) -> bool:
    """Identify FotMob's full-time who-will-win market."""
    header = market.get("header")
    translation_key = market.get("headerTranslationKey")
    normalized_header = header.strip().casefold() if isinstance(header, str) else ""
    normalized_key = (
        translation_key.strip().casefold()
        if isinstance(translation_key, str)
        else ""
    )
    return normalized_header == "1x2" or normalized_key == "who_will_win"


# Find 1x2 market for reuse in the workflow.
def find_1x2_market(odds: dict[str, Any]) -> dict[str, Any]:
    """Find one canonical 1X2 market without duplicating tab data."""
    matchfact_markets = odds.get("matchfactMarkets")
    if isinstance(matchfact_markets, list):
        # Process each available item while preserving the current workflow state.
        for market in matchfact_markets:
            if isinstance(market, dict) and is_1x2_market(market):
                return market

    odds_tab_groups = odds.get("oddsTabMarkets")
    if isinstance(odds_tab_groups, list):
        # Process each available item while preserving the current workflow state.
        for group in odds_tab_groups:
            if not isinstance(group, dict):
                continue
            markets = group.get("markets")
            if not isinstance(markets, list):
                continue
            # Process each available item while preserving the current workflow state.
            for market in markets:
                if isinstance(market, dict) and is_1x2_market(market):
                    return market

    raise OddsUnavailableError("No full-time 1X2 market is available.")


# Parse and validate decimal odds for reuse in the workflow.
def parse_decimal_odds(value: Any, selection_name: str) -> float:
    """Convert a FotMob decimal-odds value to a validated float."""
    # Validate the input before continuing with later processing.
    if isinstance(value, bool):
        raise OddsUnavailableError(
            f"Selection {selection_name!r} has invalid decimal odds."
        )
    # Validate the input before continuing with later processing.
    if not isinstance(value, (str, int, float)):
        raise OddsUnavailableError(
            f"Selection {selection_name!r} has invalid decimal odds."
        )

    # Handle expected failures with a clear, actionable message.
    try:
        decimal_odds = float(value)
    except (TypeError, ValueError) as exc:
        raise OddsUnavailableError(
            f"Selection {selection_name!r} has invalid decimal odds {value!r}."
        ) from exc

    # Validate the input before continuing with later processing.
    if not math.isfinite(decimal_odds) or decimal_odds < 1.0:
        raise OddsUnavailableError(
            f"Selection {selection_name!r} has invalid decimal odds {value!r}."
        )
    return decimal_odds


# Extract tipico 1x2 for reuse in the workflow.
def extract_tipico_1x2(payload: dict[str, Any]) -> dict[str, Any]:
    """Extract exactly one Tipico home/draw/away bookmaker record."""
    odds, provider_id = validate_tipico_payload(payload)
    market = find_1x2_market(odds)

    selections = market.get("selections")
    # Validate the input before continuing with later processing.
    if not isinstance(selections, list):
        raise OddsUnavailableError("The 1X2 market has no valid selections list.")

    required_names = {"1", "x", "2"}
    outcomes: dict[str, dict[str, Any]] = {}
    # Process each available item while preserving the current workflow state.
    for selection in selections:
        if not isinstance(selection, dict):
            continue
        name = selection.get("name")
        if not isinstance(name, str):
            continue
        normalized_name = name.strip().casefold()
        if normalized_name not in required_names:
            continue
        # Validate the input before continuing with later processing.
        if normalized_name in outcomes:
            raise OddsUnavailableError(
                f"The 1X2 market contains duplicate {normalized_name!r} selections."
            )
        outcomes[normalized_name] = selection

    missing_names = required_names - outcomes.keys()
    # Validate the input before continuing with later processing.
    if missing_names:
        raise OddsUnavailableError(
            "The 1X2 market is missing selection(s): "
            + ", ".join(sorted(missing_names))
            + "."
        )

    return {
        "bookmaker": TIPICO_DISPLAY_NAME,
        "bookmaker_source_id": provider_id,
        "home_win": parse_decimal_odds(
            outcomes["1"].get("oddsDecimal"), "1"
        ),
        "draw": parse_decimal_odds(
            outcomes["x"].get("oddsDecimal"), "x"
        ),
        "away_win": parse_decimal_odds(
            outcomes["2"].get("oddsDecimal"), "2"
        ),
    }


## Retrieve FotMob JSON through one Chrome session

Configure undetected Chrome once, inspect document status when Chrome exposes it, wait explicitly for a non-empty response body, and validate that each response is a JSON object.


In [6]:
# Handle performance log for reuse in the workflow.
def clear_performance_log(driver_instance: Any) -> None:
    """Discard status entries left by the preceding navigation."""
    # Handle expected failures with a clear, actionable message.
    try:
        driver_instance.get_log("performance")
    except Exception:
        pass


# Read performance log for reuse in the workflow.
def read_performance_log(driver_instance: Any) -> list[dict[str, Any]]:
    """Read Chrome performance entries when available."""
    # Handle expected failures with a clear, actionable message.
    try:
        return driver_instance.get_log("performance")
    except Exception:
        return []


# Find document status for reuse in the workflow.
def find_document_status(
    log_entries: list[dict[str, Any]],
    requested_url: str,
    current_url: str | None = None,
) -> int | None:
    """Find the HTTP status for the requested document navigation."""
    candidate_urls = {requested_url.rstrip("/")}
    if current_url:
        candidate_urls.add(current_url.rstrip("/"))

    observed_status: int | None = None
    # Process each available item while preserving the current workflow state.
    for entry in log_entries:
        # Handle expected failures with a clear, actionable message.
        try:
            message = json.loads(entry["message"])["message"]
            if message.get("method") != "Network.responseReceived":
                continue

            parameters = message.get("params", {})
            response = parameters.get("response", {})
            response_url = str(response.get("url", "")).rstrip("/")
            if parameters.get("type") != "Document":
                continue
            if response_url not in candidate_urls:
                continue
            observed_status = int(float(response["status"]))
        except (KeyError, TypeError, ValueError, json.JSONDecodeError):
            continue

    return observed_status


# Handle payload status for reuse in the workflow.
def reported_payload_status(payload: dict[str, Any]) -> int | None:
    """Read a numeric error status embedded in a JSON response."""
    candidates: list[Any] = [
        payload.get("status"),
        payload.get("statusCode"),
        payload.get("code"),
    ]
    error_value = payload.get("error")
    # Choose the appropriate path for the current data state.
    if isinstance(error_value, dict):
        candidates.extend(
            [
                error_value.get("status"),
                error_value.get("statusCode"),
                error_value.get("code"),
            ]
        )
    elif isinstance(error_value, str) and "404" in error_value:
        return 404

    # Process each available item while preserving the current workflow state.
    for candidate in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            status = int(str(candidate).strip())
        except (TypeError, ValueError):
            continue
        if status >= 400:
            return status
    return None


# Handle odds json for reuse in the workflow.
def retrieve_odds_json(
    driver_instance: Any, match_id: int
) -> dict[str, Any]:
    """Retrieve and parse one FotMob Tipico odds response."""
    url = ENDPOINT_TEMPLATE.format(match_id=match_id)
    clear_performance_log(driver_instance)

    # Handle expected failures with a clear, actionable message.
    try:
        driver_instance.get(url)
    except TimeoutException as exc:
        status = find_document_status(
            read_performance_log(driver_instance),
            url,
            getattr(driver_instance, "current_url", None),
        )
        # Validate the input before continuing with later processing.
        if status == 404:
            raise OddsNotFoundError(
                "FotMob returned HTTP 404; no Tipico odds are available."
            ) from exc
        raise OddsRetrievalError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(f"Chrome could not load {url}: {exc}") from exc

    status = find_document_status(
        read_performance_log(driver_instance),
        url,
        getattr(driver_instance, "current_url", None),
    )
    # Validate the input before continuing with later processing.
    if status == 404:
        raise OddsNotFoundError(
            "FotMob returned HTTP 404; no Tipico odds are available."
        )
    # Validate the input before continuing with later processing.
    if status is not None and status >= 400:
        raise OddsRetrievalError(f"FotMob returned HTTP {status} for {url}.")

    # Handle body text for reuse in the workflow.
    def nonempty_body_text(current_driver: Any) -> str | bool:
        body = current_driver.find_element(By.TAG_NAME, "body")
        body_text = body.text.strip()
        return body_text if body_text else False

    # Handle expected failures with a clear, actionable message.
    try:
        response_text = WebDriverWait(
            driver_instance, WAIT_TIMEOUT_SECONDS
        ).until(nonempty_body_text)
    except TimeoutException as exc:
        raise OddsRetrievalError(
            f"FotMob returned no readable body within "
            f"{WAIT_TIMEOUT_SECONDS} seconds."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(
            f"Chrome could not read the FotMob response body: {exc}"
        ) from exc

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise InvalidOddsJSONError(
            "FotMob did not return valid JSON "
            f"(line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise InvalidOddsJSONError(
            "The FotMob odds response must be a JSON object."
        )

    payload_status = reported_payload_status(payload)
    # Validate the input before continuing with later processing.
    if payload_status == 404:
        raise OddsNotFoundError(
            "FotMob returned HTTP 404; no Tipico odds are available."
        )
    # Validate the input before continuing with later processing.
    if payload_status is not None:
        raise OddsRetrievalError(
            f"FotMob reported HTTP {payload_status} in the response body."
        )
    return payload


## Process every match and always close Chrome

Run each match independently so an unavailable or malformed response does not stop the matchday. Browser cleanup runs even when initialization or processing raises an unexpected error.


In [7]:
options = uc.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

driver = None
results: list[dict[str, Any]] = []
# Handle expected failures with a clear, actionable message.
try:
    # Handle expected failures with a clear, actionable message.
    try:
        driver = uc.Chrome(version_main = 150,options=options, use_subprocess=True)
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
        driver.execute_cdp_cmd("Network.enable", {})
    except Exception as exc:
        raise RuntimeError(
            "Could not initialize undetected Chrome. Confirm that a compatible "
            f"Chrome installation is available. Original error: {exc}"
        ) from exc

    total_matches = len(matches)
    # Process each available item while preserving the current workflow state.
    for match_number, match in enumerate(matches, start=1):
        match_id = match["match_id"]
        home_team = match["home_team"]
        away_team = match["away_team"]
        result: dict[str, Any] = {
            "match_id": match_id,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers": [],
            "issues": [],
        }

        print(
            f"[{match_number}/{total_matches}] {home_team} vs {away_team} "
            f"(match_id={match_id})"
        )

        # Handle expected failures with a clear, actionable message.
        try:
            payload = retrieve_odds_json(driver, match_id)
            bookmaker = extract_tipico_1x2(payload)
            result["bookmakers"].append(bookmaker)
            print("  Extracted Tipico full-time 1X2 odds.")
        except OddsNotFoundError as exc:
            result["issues"].append(str(exc))
            print(f"  404: {exc}")
        except OddsNotebookError as exc:
            result["issues"].append(f"{type(exc).__name__}: {exc}")
            print(f"  Issue: {type(exc).__name__}: {exc}")
        except Exception as exc:
            result["issues"].append(
                f"Unexpected error: {type(exc).__name__}: {exc}"
            )
            print(f"  Unexpected error: {type(exc).__name__}: {exc}")

        results.append(result)
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
            print("Chrome driver closed.")
        except Exception as cleanup_error:
            print(f"Chrome driver shutdown warning: {cleanup_error}")
        finally:
            driver = None

print("Processing complete.")


[1/9] Bayern München vs VfB Stuttgart (match_id=5881143)
  Extracted Tipico full-time 1X2 odds.
[2/9] Elversberg vs Bayer Leverkusen (match_id=5881146)
  Extracted Tipico full-time 1X2 odds.
[3/9] 1. FC Köln vs Hoffenheim (match_id=5881147)
  Extracted Tipico full-time 1X2 odds.
[4/9] Mainz 05 vs Paderborn (match_id=5881149)
  Extracted Tipico full-time 1X2 odds.
[5/9] RB Leipzig vs Borussia Mönchengladbach (match_id=5881150)
  Extracted Tipico full-time 1X2 odds.
[6/9] Union Berlin vs Eintracht Frankfurt (match_id=5881151)
  Extracted Tipico full-time 1X2 odds.
[7/9] Borussia Dortmund vs Hamburger SV (match_id=5881145)
  Extracted Tipico full-time 1X2 odds.
[8/9] Freiburg vs Werder Bremen (match_id=5881148)
  Extracted Tipico full-time 1X2 odds.
[9/9] Augsburg vs Schalke 04 (match_id=5881144)
  Extracted Tipico full-time 1X2 odds.
Chrome driver closed.
Processing complete.


## Inspect, summarize, and save the matchday

Display the complete result, print concise counts, and save readable UTF-8 JSON in `outputs/fotmob/odds`.


In [8]:
pprint(results, sort_dicts=False)

matches_with_odds = sum(bool(result["bookmakers"]) for result in results)
matches_without_odds = len(results) - matches_with_odds
matches_with_issues = sum(bool(result["issues"]) for result in results)
bookmaker_records = sum(len(result["bookmakers"]) for result in results)

print("\nSummary")
print(f"  Bundesliga matchday:       {matchday}")
print(f"  Total input matches:       {len(matches)}")
print(f"  Matches with Tipico odds:  {matches_with_odds}")
print(f"  Matches without odds:      {matches_without_odds}")
print(f"  Matches with issues:       {matches_with_issues}")
print(f"  Tipico bookmaker records:  {bookmaker_records}")

# Handle expected failures with a clear, actionable message.
try:
    output_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
except OSError as exc:
    raise OSError(f"Could not save {output_path}: {exc}") from exc

print(f"Saved {len(results)} match result(s) to {output_path.resolve()}.")


[{'match_id': 5881143,
  'home_team': 'Bayern München',
  'away_team': 'VfB Stuttgart',
  'bookmakers': [{'bookmaker': 'Tipico',
                  'bookmaker_source_id': 171,
                  'home_win': 1.25,
                  'draw': 7.0,
                  'away_win': 9.0}],
  'issues': []},
 {'match_id': 5881146,
  'home_team': 'Elversberg',
  'away_team': 'Bayer Leverkusen',
  'bookmakers': [{'bookmaker': 'Tipico',
                  'bookmaker_source_id': 171,
                  'home_win': 5.2,
                  'draw': 4.6,
                  'away_win': 1.55}],
  'issues': []},
 {'match_id': 5881147,
  'home_team': '1. FC Köln',
  'away_team': 'Hoffenheim',
  'bookmakers': [{'bookmaker': 'Tipico',
                  'bookmaker_source_id': 171,
                  'home_win': 3.0,
                  'draw': 3.6,
                  'away_win': 2.25}],
  'issues': []},
 {'match_id': 5881149,
  'home_team': 'Mainz 05',
  'away_team': 'Paderborn',
  'bookmakers': [{'bookmaker': 'Tipico',
 